[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S2_hoja_metricas.ipynb)

# Hoja de trabajo: ¿cuántas bicis salen mañana a las 8?

**Al terminar podrás** dar un pronóstico para tu estación con el número que lo respalda (MAE, RMSE y R²),
y explicar en tres renglones qué tan confiable es, sin usar una métrica que no sirve con ceros.

Sección 1 de S2, de 15 a 20 minutos en sala. No se entrega ni se califica.

---

**De:** Coordinación de operaciones, MiBici<br>
**Asunto:** pronóstico para mañana a las 8:00

Buen día. Mañana, lunes 3 de agosto, vamos a planear el acomodo de bicis de la mañana.
Necesitamos, para su estación: **¿cuántas bicis van a salir a las 8:00 y qué tan confiable es ese número?**
Tenemos dos propuestas del equipo: la regla "las mismas que hace una semana" y la recta que ajustaron ayer.
Usen julio para medirlas. Respondan en tres renglones: el número, qué tan seguro es y qué recomiendan.

---

| Paso | Qué hacen | Minutos |
|---|---|---|
| 1. Los datos | correr y escoger estación | 3 |
| 2. El modelo | cambiar **una línea**: `estacion = 51` | 4 |
| 3. La medición | leer MAE, RMSE y R²; detectar el MAPE que truena | 6 |
| 4. La respuesta | tres renglones con sus números | 4 |

**Cómo trabajar.** En sala, una persona comparte pantalla y las demás dictan. Cada quien escoge una estación distinta;
si no alcanza el tiempo, hagan todo con una y comparen al final. Las preguntas de opción múltiple se contestan en la hoja, con la letra.

Corran las celdas de abajo hasta que aparezca `✅ entorno listo`.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'seaborn': 'seaborn==0.13.2',
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from dammg_utils import COLORES

du.estilo_dammg()

## Paso 1. Los datos (3 min)

Cada fila es una estación en una hora. La columna que queremos predecir es `departures`: cuántas bicis salieron de verdad.
`departures_lag_168h` son las salidas de esa misma estación, a esa misma hora, hace una semana (168 horas).

- **train** (octubre a junio): con esto se ajusta la recta.
- **julio** (validación): con esto medimos y escogemos. Es lo que le enseñamos a operaciones.
- agosto y septiembre (prueba): no se tocan; son el examen final del curso.

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
train = df[df.particion == "train"]   # octubre a junio: para ajustar
julio = df[df.particion == "val"]     # julio: para medir y escoger

# La recta de ayer: salidas de hoy a partir de las de hace una semana, ajustada solo con train
recta = LinearRegression().fit(train[["departures_lag_168h"]], train.departures)
print(f"recta: salidas = {recta.coef_[0]:.2f} × (hace una semana) + {recta.intercept_:.2f}")

Las 40 estaciones del curso. `salidas_por_hora` es el promedio de julio y `horas_en_0` la fracción de horas de julio
en que no salió ninguna bici. Escojan una: una con mucho movimiento, una tranquila, la que les quede cerca.

**Ojo con las estaciones cerradas.** Si su estación tiene muchos días seguidos en 0, puede que haya estado cerrada
(obras, mantenimiento); escojan otra. En esos días cualquier propuesta que diga 0 "acierta" sin mérito
y las métricas salen mejores de lo que son. En el Paso 2 una celda lo revisa por ustedes.

In [ ]:
estaciones = julio.groupby("station_id").agg(
    capacidad=("capacity", "first"),                         # anclajes de la estación
    salidas_por_hora=("departures", "mean"),                  # promedio de julio
    horas_en_0=("departures", lambda s: (s == 0).mean()),     # fracción de horas sin salidas
).round(2)
estaciones.sort_values("salidas_por_hora", ascending=False)

## Paso 2. El modelo (4 min)

**Cambien el 51 por el número de su estación** y corran. Es la única línea que se edita en toda la hoja.

La celda calcula las dos propuestas para cada hora de julio y luego el pronóstico de mañana.
Para el lunes 3 de agosto a las 8:00, "hace una semana" es el lunes 27 de julio a las 8:00, que sí está en julio.
Como mañana es lunes, las 8:00 se miden solo de lunes a viernes: los sábados y domingos a esa hora casi no sale nadie.

In [ ]:
estacion = 51   # <- cambien este número por el de su estación

assert estacion in estaciones.index, (f"La estación {estacion} no está en la lista; "
                                      f"usen una de: {sorted(estaciones.index.tolist())}")
mia = julio[julio.station_id == estacion].copy()
mia["regla"] = mia.departures_lag_168h                               # las mismas que hace una semana
mia["recta"] = recta.predict(mia[["departures_lag_168h"]])            # la recta de ayer
ocho = mia[(mia.hour_of_day == 8) & (mia.is_weekend == 0)]           # las 8:00 de lunes a viernes de julio

# Mañana, lunes 3 de agosto a las 8:00: se calcula con el lunes 27 de julio a las 8:00
base = mia.loc[mia.hour == "2026-07-27 08:00", "departures"].iloc[0]
manana = {"regla": base, "recta": recta.predict(pd.DataFrame({"departures_lag_168h": [base]}))[0]}
print(f"Estación {estacion}. El lunes 27 de julio a las 8:00 salieron {base} bicis.")
print(f"Pronóstico para el lunes 3 de agosto a las 8:00:  regla {manana['regla']:.0f}   recta {manana['recta']:.1f}")

In [ ]:
# ¿Estuvo cerrada? Buscamos la racha más larga de días seguidos de julio sin ninguna salida
por_dia = mia.groupby(mia.hour.dt.date).departures.sum()   # salidas de cada día de julio
racha, mas_larga = 0, 0
for salidas in por_dia:
    racha = racha + 1 if salidas == 0 else 0                 # se reinicia en cuanto sale una bici
    mas_larga = max(mas_larga, racha)
print(f"Estación {estacion}: {mas_larga} días seguidos de julio en 0.")
if mas_larga >= 3:
    print("Aviso: muchos días seguidos en 0: puede que haya estado cerrada. Escojan otra estación y vuelvan a correr.")

In [ ]:
# Las 8:00 de lunes a viernes de julio: lo real contra las dos propuestas
largo = ocho.melt(id_vars="hour", value_vars=["departures", "regla", "recta"], var_name="serie", value_name="bicis")
largo["serie"] = largo.serie.replace({"departures": "real"})
plt.figure(figsize=(10, 4))
ax = sns.lineplot(data=largo, x="hour", y="bicis", hue="serie", marker="o",
                  palette={"real": COLORES["tinta"], "regla": COLORES["w"], "recta": COLORES["yhat"]})
ax.set(xlabel="julio, de lunes a viernes a las 8:00", ylabel="bicis que salieron", title=f"Estación {estacion}")
plt.xticks(rotation=30)
plt.show()

## Paso 3. La medición (6 min)

Tres números por propuesta, medidos en julio. Recuerden cómo se lee cada uno:

- **MAE**: por cuántas bicis fallamos en una hora típica, sin importar el signo. En bicis.
- **RMSE**: también en bicis, pero el cuadrado castiga más los errores grandes. Si sale mucho mayor que el MAE,
  hay pocas horas con errores muy grandes.
- **R²**: cuánto mejor que decir siempre el promedio de julio de la estación. 1 es perfecto, 0 es igual que el promedio,
  negativo es peor que el promedio. No tiene unidades.

Para la pregunta de operaciones agregamos dos columnas solo con las 8:00 de lunes a viernes (mañana es lunes):
el MAE a esa hora, y el error que **no se pasó en 8 de cada 10 días laborables** (si dice 6, en 8 de cada 10
mañanas laborables de julio fallamos por 6 bicis o menos).

In [ ]:
def mide(d, col):
    """Las métricas de una propuesta (col) en un pedazo de julio (d)."""
    return {"MAE": mean_absolute_error(d.departures, d[col]),
            "RMSE": root_mean_squared_error(d.departures, d[col]),
            "R²": r2_score(d.departures, d[col])}


tabla = pd.DataFrame({col: mide(mia, col) for col in ["regla", "recta"]}).T
tabla["MAE a las 8"] = [mean_absolute_error(ocho.departures, ocho[c]) for c in ["regla", "recta"]]
tabla["8 de 10 días laborables, error de hasta"] = [(ocho[c] - ocho.departures).abs().quantile(0.8)
                                                   for c in ["regla", "recta"]]
tabla.round(2)

In [ ]:
# MAE contra RMSE de las dos propuestas, en barras
barras = tabla[["MAE", "RMSE"]].reset_index(names="propuesta").melt(id_vars="propuesta", var_name="métrica",
                                                                     value_name="bicis")
ax = sns.barplot(data=barras, x="métrica", y="bicis", hue="propuesta",
                 palette={"regla": COLORES["w"], "recta": COLORES["yhat"]}, edgecolor=COLORES["tinta"])
ax.set(title=f"Estación {estacion}, julio completo", xlabel="")
plt.show()

**1. En su tabla, ¿cómo se le dice el MAE de la recta a operaciones?**

- a) En una hora típica de julio, la recta falla por esa cantidad de bicis, para arriba o para abajo
- b) La recta acierta ese porcentaje de las horas
- c) Es el error de la peor hora de julio
- d) Es la proporción de la variación que explica la recta

**Su respuesta (doble clic para escribir):** ___

**2. Si en su tabla el RMSE es mucho más grande que el MAE (en la 51, la recta: 9.2 contra 5.7), ¿qué les dice?**

- a) Que el modelo está mal programado
- b) Que hay pocas horas con errores muy grandes que el cuadrado infla
- c) Que el MAE está mal calculado
- d) Que julio tiene pocas horas

**Su respuesta (doble clic para escribir):** ___

**Un caso más.** Finanzas pidió el error "en porcentaje", el **MAPE**: el promedio de |ŷ − y| / y.
Corran la celda y miren el número.

In [ ]:
mape = mean_absolute_percentage_error(mia.departures, mia.recta)
print(f"MAPE de la recta en julio: {mape:,.0%}")          # scikit-learn lo da como fracción; :% lo muestra en porcentaje
print(f"horas de julio con 0 salidas en la estación {estacion}: {(mia.departures == 0).mean():.0%}")

con_salidas = mia[mia.departures > 0]                     # quitando las horas en 0
print(f"MAPE sin las horas en 0: {mean_absolute_percentage_error(con_salidas.departures, con_salidas.recta):,.0%}")

**3. El MAPE salió de más de mil billones por ciento. ¿Qué pasó?**

- a) La recta es pésima en esta estación
- b) Hay horas con 0 salidas y el MAPE divide entre lo real
- c) Hay que dividir entre 100
- d) Es un error de scikit-learn

**Su respuesta (doble clic para escribir):** ___

**4. En las estaciones 32, 50, 196 y 268 la regla tiene R² negativo en julio. ¿Qué significa?**

- a) Que la regla se equivoca hacia abajo
- b) Que la regla es peor que decir siempre el promedio de julio de esa estación
- c) Que hay un error de cálculo: R² va de 0 a 1
- d) Que la estación tiene menos salidas que la semana pasada

**Su respuesta (doble clic para escribir):** ___

## Paso 4. La respuesta (4 min)

Corran la celda: arma el borrador con sus números. Luego escriban su respuesta en la celda de texto de abajo,
en tres renglones, con sus palabras. Escojan la propuesta con menor **MAE a las 8**: es la que mejor contesta la pregunta.

In [ ]:
mejor = tabla["MAE a las 8"].idxmin()                       # la propuesta que menos falla a las 8:00
mae8 = tabla.loc[mejor, "MAE a las 8"]                       # su error promedio a las 8:00, lunes a viernes
hasta = tabla.loc[mejor, "8 de 10 días laborables, error de hasta"]
print(f"1. El número: mañana a las 8:00 esperamos unas {manana[mejor]:.0f} salidas en la estación {estacion} ({mejor}).")
print(f"2. Qué tan seguro: en julio, a las 8:00 de lunes a viernes, fallamos en promedio por {mae8:.1f} bicis, "
      f"y en 8 de cada 10 días laborables por {hasta:.0f} o menos.")
# El pronóstico más el MAE, redondeado: no salen fracciones de bici. Suponemos que a las 8:00
# la estación está llena y que no llegan bicis durante la hora.
plan = round(manana[mejor] + mae8)
anclajes = estaciones.loc[estacion, "capacidad"]             # cuántas bicis caben en la estación
print(f"3. Recomendación y riesgo: planear para unas {plan} salidas (el pronóstico más el MAE); la estación "
      f"tiene {anclajes} anclajes" + (", así que habrá que reabastecerla durante la hora" if plan > anclajes else "")
      + f". En 2 de cada 10 mañanas laborables de julio fallamos por más de {hasta:.0f}.")

**Su respuesta a operaciones** (doble clic para escribir):

1. **El número:** mañana, lunes 3 de agosto, a las 8:00, en la estación ___ esperamos unas ___ salidas.
2. **Qué tan seguro:** en julio, a esa hora y de lunes a viernes, fallamos en promedio por ___ bicis; en 8 de cada 10 días laborables, por ___ o menos.
3. **Recomendación y riesgo:** ___ (para cuántas salidas planear, si caben en los anclajes, y qué pasa en los días en que fallamos más).

Dejen su respuesta escrita en la hoja; al volver, la revisamos juntos.